In [ ]:
from autogen import AssistantAgent,UserProxyAgent,GroupChat,GroupChatManager
from autogen_agentchat.agents import AssistantAgent,UserProxyAgent
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
import os

In [ ]:
config_list = [{
    "model": "openai/gpt-oss-120b",
    "api_key": os.environ.get("GROQ_API_KEY"),
    "api_type": "groq"
}]

In [ ]:
from typing import Dict, List

def travel_data_tool(destination: str) -> Dict:
    data = {
        "Goa": {
            "attractions": ["Baga Beach", "Dudhsagar Falls", "Fort Aguada"],
            "hotels": [
                {"name": "Hotel Sea View", "price_per_night": 2500},
                {"name": "Goa Paradise Resort", "price_per_night": 3500}
            ],
            "transport_avg_cost": 4000,
            "daily_food_cost": 800
        },
        "Ooty": {
            "attractions": ["Ooty Lake", "Botanical Garden", "Doddabetta Peak"],
            "hotels": [
                {"name": "Hill View Inn", "price_per_night": 2000},
                {"name": "Green Valley Resort", "price_per_night": 3000}
            ],
            "transport_avg_cost": 3000,
            "daily_food_cost": 700
        }
    }

    return data.get(destination, {"error": "Destination not found"})


In [ ]:
from autogen import AssistantAgent, GroupChat, UserProxyAgent,GroupChatManager
planner_agent = AssistantAgent(
    name="Planner",
    system_message="You are a Holiday Planner Agent. You help users plan their holidays by providing recommendations on destinations, activities, and accommodations based on their preferences and budget.",
    llm_config = {
        "config_list":config_list
    }
)

Researcher_agent = AssistantAgent(
    name="Research_Agent",
    system_message="""
    You are a travel research expert.
    When destination is provided, call the travel_data_tool and verify using travel_data_tool only be answer.
    Always return structured data.
    """,
    llm_config={
        "config_list":config_list
    }
)


Researcher_agent.register_function(
    function_map={
        "travel_data_tool": travel_data_tool
    }
)


user_agent = UserProxyAgent(name="User"
                            ,human_input_mode="NEVER",
                            code_execution_config=False)
group_chat = GroupChat(agents=[planner_agent, Researcher_agent, user_agent],max_round=8)
group_chat_manager = GroupChatManager(group_chat,llm_config={
    "config_list":config_list})

In [ ]:
res = user_agent.initiate_chat(
    group_chat_manager,
    message="Hi, I am planning a holiday and would like some recommendations on destinations, activities, and accommodations and I prefer beach destinations in Goa and return only tool information with waterfalls."
)
